# HW6 Playground

Fill in TODOs as you work through the assignment.
Implement the required sections in `hw6_loader.py`, `model.py`, and `train.py`, and use this notebook to explore both datasets, compare models, and interpret embeddings.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import random

import numpy as np
import pandas as pd
import torch

from hw6_loader import HW6DataLoader
from model import RGCNLinkPredictor, GCNLinkPredictor
from train import train, evaluate
from utils import (
    print_graph_stats,
    plot_node_degree_distribution,
    plot_edge_type_distribution,
    plot_embeddings,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loader = HW6DataLoader()

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


## Split Paths and Loaders

In [ ]:
split_roots = {
    "drug_disease": os.path.join("..", "data", "drug_disease_splits"),
    "drug_drug": os.path.join("..", "data", "drug_drug_splits"),
}

def load_split_graph(dataset_name, embedding_dim=16, feature_type="random", val_ratio=0.2, seed=42):
    split_dir = split_roots[dataset_name]
    train_csv = os.path.join(split_dir, "train.csv")
    test_csv = os.path.join(split_dir, "test.csv")

    # if dataset_name == "drug_disease":
    #     return loader.get_drug_disease_data_split(...)        
    # if dataset_name == "drug_drug":
    #     return loader.get_drug_drug_data_split(...)


## Inspect Graph Structure

In [ ]:
# TODO: Inspect both datasets.
# for dataset_name in ["drug_disease", "drug_drug"]:
#     data = load_split_graph(dataset_name, embedding_dim=16, feature_type="random")
#     print("\nDataset:", dataset_name)
#     print_graph_stats(...)
#     plot_node_degree_distribution(...)
#     plot_edge_type_distribution(...)
#     if hasattr(data, "node_names"):
#         print("Example nodes:", data.node_names[:5])
#     if hasattr(data, "relation_names"):
#         print("Relation labels:", data.relation_names)


## Model and Hyperparameter Sweeps

In [ ]:
configs = [
    {"dataset": "drug_disease", "model": "gcn", "embedding_dim": 8, "num_layers": 2, "feature_type": "random", "epochs": 20},
    {"dataset": "drug_disease", "model": "rgcn", "embedding_dim": 16, "num_layers": 3, "feature_type": "one-hot", "epochs": 20},
    {"dataset": "drug_drug", "model": "gcn", "embedding_dim": 8, "num_layers": 2, "feature_type": "random", "epochs": 20},
    {"dataset": "drug_drug", "model": "rgcn", "embedding_dim": 16, "num_layers": 3, "feature_type": "one-hot", "epochs": 20},
]
results = []

# TODO: for each config:
# 1. call set_seed(1) or another fixed seed
# 2. load the requested dataset with config["embedding_dim"] and config["feature_type"]
# 3. initialize GCNLinkPredictor or RGCNLinkPredictor with config["num_layers"]
# 4. train for config["epochs"] epochs
# 5. evaluate on validation/test edges and append a summary row to results


## Embedding Visualization and Interpretation

In [ ]:
# TODO: choose one trained run from each dataset and visualize the embeddings.
# Use the train-only message-passing graph, not the full graph, when you
# compute embeddings for evaluation or visualization.
# with torch.no_grad():
#     embeddings = model(
#         data.x.to(device),
#         data.message_passing_edge_index.to(device),
#         data.message_passing_edge_type.to(device),
#     )
# plot_embeddings(embeddings.cpu().numpy(), method="pca")
# plot_embeddings(embeddings.cpu().numpy(), method="tsne")

# TODO: connect the plots back to biomedical context.
# Useful checks:
# - inspect data.node_names for points in a cluster or far-away outliers
# - inspect data.relation_names when comparing GCN vs R-GCN runs
# - if you stored node types, color the points by drug vs disease
# - compare whether relation-aware message passing changes cluster separation


## Optional Extension: Description-Based Features


In [ ]:
# Optional TODO: extend make_node_features(...) in hw6_loader.py so
# feature_type="pretrained" uses node descriptions.
#
# One simple route is to build TF-IDF features from the text:
#
# from sklearn.feature_extraction.text import TfidfVectorizer
#
# descriptions = getattr(data, "node_descriptions", [])
# vectorizer = TfidfVectorizer(max_features=256, stop_words="english")
# desc_features = vectorizer.fit_transform(descriptions).toarray()
# x = torch.tensor(desc_features, dtype=torch.float)
#
# After you add the "pretrained" branch, compare it against the
# required feature types:
# for feature_type in ["random", "one-hot", "pretrained"]:
#     data = load_split_graph("drug_disease", embedding_dim=16, feature_type=feature_type)
#     ... train / evaluate / log metrics ...


## PCA/t-SNE Interpretation with Node Labels


In [ ]:
# TODO: after computing embeddings, inspect and label a small set of
# representative points.
#
# Start with the simple helper for an unlabeled overview:
# plot_embeddings(embeddings.cpu().numpy(), method="pca")
# plot_embeddings(embeddings.cpu().numpy(), method="tsne")
#
# If you want node-type colors or text labels, do that explicitly here in the
# notebook so the logic stays easy to read.
#
# Example: color points by node type on a PCA plot.
# import matplotlib.pyplot as plt
# from sklearn.decomposition import PCA
# reduced = PCA(n_components=2).fit_transform(embeddings.cpu().numpy())
# node_types = np.array(getattr(data, "node_types", ["node"] * data.num_nodes))
# colors = np.where(node_types == "drug", "tab:blue", "tab:orange")
# plt.figure(figsize=(6, 6))
# plt.scatter(reduced[:, 0], reduced[:, 1], c=colors, s=8)
# plt.show()
#
# To annotate a few specific nodes, start from their metadata:
# nodes = [0, 5, 12]  # replace with points you want to inspect
# plt.figure(figsize=(6, 6))
# plt.scatter(reduced[:, 0], reduced[:, 1], c=colors, s=8)
# for idx in nodes:
#     label = data.node_display_names[idx] if hasattr(data, "node_display_names") else data.node_names[idx]
#     plt.annotate(label, (reduced[idx, 0], reduced[idx, 1]), fontsize=8)
# plt.show()
#
# Then relate those labeled points back to drug/disease identity,
# nearby relation types, and whether GCN vs R-GCN separates them differently.
